# RMIT_SGS_HaiQuayXe_C1

**Version 3: final refit on all 14,000 labelled training rows.** Approved backbone: `FacebookAI/xlm-roberta-base`. Team HaiQuayXe, RMIT SGS.

The development experiment v2 selected 512-token inputs, three epochs, learning rate 2e-5 and effective batch 16. Its separate 11,200/2,800 split achieved macro-F1 **0.9425011698287723**. That score belongs to the earlier development model; **this refitted model has no held-out validation score**. We now initialize again from the approved base checkpoint and train on all supplied labelled rows, with no checkpoint selection or holdout-based tuning.

Test labels are unavailable and never used. Prompts are untrusted data and are only tokenized, never executed or sent elsewhere. This private GPU notebook uses attached local competition/model inputs with internet disabled. It never automatically consumes a competition submission attempt.

**Output:** `/kaggle/working/submission.csv`, exactly `id,label`, 6,000 rows. The final remote model, provenance and training history are saved under `/kaggle/working/haiquayxe_c1`. The team must also deliver notebook code according to the organisers' requirements.


## 1. Start a fresh GPU session
Run cells in order, or choose **Run All**. If you already imported PyTorch in this session before changing GPU visibility, restart the session first. Competition prompts are untrusted text: we tokenize them as data, never execute their instructions or send them to an external service.

In [ ]:
import os
import sys

# This must happen before importing torch. The second T4, when present, stays unused.
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

import gc
import hashlib
import inspect
import json
import math
import re
import time
import unicodedata
from pathlib import Path

import datasets
import numpy as np
import pandas as pd
import torch
import transformers
from datasets import Dataset
from transformers import (
    AutoConfig,
    AutoModelForSequenceClassification,
    AutoTokenizer,
    DataCollatorWithPadding,
    Trainer,
    TrainingArguments,
    set_seed,
)

if not torch.cuda.is_available():
    raise RuntimeError(
        "No GPU is available. In Kaggle notebook settings, choose a GPU accelerator "
        "(for example T4 x2), restart the session, then Run All. Do not train on CPU."
    )
if torch.cuda.device_count() != 1:
    raise RuntimeError(
        "PyTorch saw multiple GPUs before this notebook configured visibility. "
        "Restart the session and run this cell first."
    )
print(
    {
        "python": sys.version.split()[0],
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
        "gpu": torch.cuda.get_device_name(0),
        "gpu_memory_GB": round(torch.cuda.get_device_properties(0).total_memory / 2**30, 1),
    }
)

## 2. Selected configuration and attached inputs
This final refit keeps the validated v2 configuration: first 512 tokens, three epochs, batch 4 with four accumulated batches, learning rate 2e-5, seed 42, standard cross-entropy. One T4 is used. No setting is selected using the hidden test set.


In [ ]:
SEED = 42
LABELS = ["BENIGN", "JAILBREAK", "PROMPT_INJECTION"]
L2I = {label: index for index, label in enumerate(LABELS)}
I2L = {index: label for label, index in L2I.items()}
LANGUAGES = {"pt", "fr", "it", "en"}

MAX_LENGTH = 512
EPOCHS = 3
LEARNING_RATE = 2e-5
TRAIN_BATCH_SIZE = 4
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION = 4

DATA_PATH = "/kaggle/input/competitions/rmit-hackathon-2026"
MODEL_PATH = "/kaggle/input/models/aisuko/xlm-roberta-base/transformers/default/1"
WORK = Path("/kaggle/working/haiquayxe_c1")
WORK.mkdir(parents=True, exist_ok=True)
set_seed(SEED)


def locate_data(preferred):
    required = {"train.csv", "test.csv", "sample_submission.csv"}
    candidates = [Path(preferred), Path("/kaggle/input/rmit-hackathon-2026")]
    for path in candidates:
        if all((path / name).is_file() for name in required):
            return path
    found = sorted(
        {
            path.parent
            for path in Path("/kaggle/input").rglob("train.csv")
            if all((path.parent / name).is_file() for name in required)
        }
    )
    if len(found) != 1:
        raise FileNotFoundError(
            "Attach the RMIT Hackathon 2026 competition data. If multiple datasets "
            f"match, set DATA_PATH explicitly. Matching directories: {found}"
        )
    return found[0]


def locate_model(preferred):
    preferred = Path(preferred)
    if (preferred / "config.json").is_file():
        return preferred
    found = []
    for config_path in Path("/kaggle/input").rglob("config.json"):
        if "xlm-roberta-base" not in str(config_path).lower():
            continue
        try:
            config = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        if config.get("model_type") == "xlm-roberta":
            found.append(config_path.parent)
    if len(found) != 1:
        raise FileNotFoundError(
            "Attach the organiser's XLM-RoBERTa Base model input. "
            f"Set MODEL_PATH explicitly if needed. Matching directories: {found}"
        )
    return found[0]


DATA = locate_data(DATA_PATH)
MODEL = locate_model(MODEL_PATH)
backbone_config = AutoConfig.from_pretrained(MODEL, local_files_only=True)
if (
    backbone_config.model_type != "xlm-roberta"
    or backbone_config.hidden_size != 768
    or backbone_config.num_hidden_layers != 12
):
    raise ValueError("This notebook expects the approved XLM-RoBERTa Base checkpoint.")
print("Data:", DATA)
print("Approved model:", MODEL)
print("Effective training batch:", TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION)

## 3. Validate data and assign all labelled rows to training
All 14,000 supplied training examples are used. The manifest records their IDs, languages and labels; no evaluation subset is created. Duplicate diagnostics do not change original prompt text.


In [ ]:
train = pd.read_csv(DATA / "train.csv", dtype=str, keep_default_na=False)
test = pd.read_csv(DATA / "test.csv", dtype=str, keep_default_na=False)
sample = pd.read_csv(DATA / "sample_submission.csv", dtype=str, keep_default_na=False)


def require(condition, message):
    if not condition:
        raise ValueError(message)


require(set(train.columns) == {"id", "text", "language", "label"}, "Unexpected train schema.")
require(set(test.columns) == {"id", "text", "language"}, "Unexpected test schema.")
require(list(sample.columns) == ["id", "label"], "Sample columns must be id,label.")
require(len(train) == 14000 and len(test) == 6000, "Expected 14,000 train and 6,000 test rows.")
require(len(sample) == len(test), "Sample row count does not match test.")
for frame_name, frame in [("train", train), ("test", test), ("sample", sample)]:
    require(frame["id"].str.strip().ne("").all(), f"Blank ID in {frame_name}.")
    require(frame["id"].is_unique, f"Duplicate ID in {frame_name}.")
for frame_name, frame in [("train", train), ("test", test)]:
    require(frame["text"].str.strip().ne("").all(), f"Blank prompt in {frame_name}.")
    require(set(frame["language"]) <= LANGUAGES, f"Unknown language in {frame_name}.")
require(set(train["label"]) == set(LABELS), "Unexpected or missing training class.")
require(set(test["id"]) == set(sample["id"]), "Sample/test IDs do not match.")
require(set(train["id"]).isdisjoint(test["id"]), "Train and test IDs overlap.")


def text_group(text):
    normalized = unicodedata.normalize("NFKC", text).casefold()
    normalized = re.sub(r"\s+", " ", normalized).strip()
    return hashlib.sha256(normalized.encode("utf-8")).hexdigest()


train["_group"] = train["text"].map(text_group)
train["_label_id"] = train["label"].map(L2I)
tr = train.reset_index(drop=True)
split_method = "full-data refit; no held-out evaluation"
require(len(tr) == 14000, "Refit must use all 14,000 labelled rows.")
print("Fit rows:", len(tr), "Held-out rows: 0", "Test rows:", len(test))
print("Normalized duplicate training rows:", int(train["_group"].duplicated().sum()))
print(
    "Normalized duplicate groups with conflicting labels:",
    int(train.groupby("_group")["label"].nunique().gt(1).sum()),
)
print(tr.groupby(["language", "label"]).size().rename("fit").to_string())
tr[["id", "language", "label"]].assign(split="fit").to_csv(WORK / "split_manifest.csv", index=False)

## 4. Tokenize with the selected 512-token limit
The training-only token-length sample is diagnostic. Original prompt text remains unchanged; tokenization truncates to the first 512 tokens, exactly as in development v2.


In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL, local_files_only=True, use_fast=True)
length_sample = train["text"].sample(n=min(1000, len(train)), random_state=SEED).tolist()
length_encoding = tokenizer(
    length_sample,
    truncation=False,
    padding=False,
    return_attention_mask=False,
    return_token_type_ids=False,
    verbose=False,
)
lengths = np.array([len(tokens) for tokens in length_encoding["input_ids"]])
truncation_diagnostics = {
    "sample_size": len(length_sample),
    "max_length": MAX_LENGTH,
    "fraction_longer_than_max_length": float((lengths > MAX_LENGTH).mean()),
    "median_tokens": float(np.median(lengths)),
    "p95_tokens": float(np.quantile(lengths, 0.95)),
}
print("Training-only token-length sample:", truncation_diagnostics)
del length_encoding, length_sample


def make_dataset(frame, labelled=True):
    values = {"text": frame["text"].tolist()}
    if labelled:
        values["labels"] = frame["label"].map(L2I).astype(int).tolist()
    dataset = Dataset.from_dict(values)
    return dataset.map(
        lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
        batched=True,
        remove_columns=["text"],
        load_from_cache_file=False,
    )


ds_tr = make_dataset(tr)
ds_te = make_dataset(test, labelled=False)
collator = DataCollatorWithPadding(tokenizer=tokenizer, pad_to_multiple_of=8)
print("Tokenized rows (fit / test):", len(ds_tr), len(ds_te))

## 5. Load the approved model and test a training batch
Warnings about missing `classifier.*` weights are expected: the three-class head is new and must be trained. Unused masked-language-model or pooler weights are also expected when changing the model's task. A successful load is not a trained classifier.

The smoke test performs a forward/backward pass on a batch of the longest tokenized training examples, without updating weights. If CUDA runs out of memory, restart the session, reduce `TRAIN_BATCH_SIZE` to 2 and `EVAL_BATCH_SIZE` to 4, and increase `GRADIENT_ACCUMULATION` to 8. Never fall back silently to CPU.

In [ ]:
set_seed(SEED)
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL,
    local_files_only=True,
    num_labels=len(LABELS),
    id2label=I2L,
    label2id=L2I,
)
model.to("cuda")
model.train()
longest = sorted(range(len(ds_tr)), key=lambda index: len(ds_tr[index]["input_ids"]), reverse=True)[
    :TRAIN_BATCH_SIZE
]
smoke_batch = {
    key: value.to("cuda") for key, value in collator([ds_tr[i] for i in longest]).items()
}
with torch.autocast(device_type="cuda", dtype=torch.float16):
    smoke_output = model(**smoke_batch)
    smoke_loss = smoke_output.loss
require(bool(torch.isfinite(smoke_loss).item()), "Non-finite loss in the smoke test.")
smoke_loss.backward()
print("Forward/backward smoke test passed. Initial untrained loss:", round(smoke_loss.item(), 4))
model.zero_grad(set_to_none=True)
del smoke_batch, smoke_output, smoke_loss
_gc_collected = gc.collect()
torch.cuda.empty_cache()
set_seed(SEED)

## 6. Refit from the approved base checkpoint
Train for exactly three epochs using all 14,000 labelled examples. Evaluation and intermediate model checkpoints are disabled. The final epoch model is exported; there is no best-checkpoint selection and no validation-score claim for this refit.


In [ ]:
batches_per_epoch = math.ceil(len(ds_tr) / TRAIN_BATCH_SIZE)
optimizer_steps = math.ceil(batches_per_epoch / GRADIENT_ACCUMULATION) * EPOCHS
warmup_steps = max(1, math.ceil(optimizer_steps * 0.06))

argument_values = dict(
    output_dir=str(WORK / "checkpoints"),
    num_train_epochs=EPOCHS,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=TRAIN_BATCH_SIZE,
    per_device_eval_batch_size=EVAL_BATCH_SIZE,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION,
    weight_decay=0.01,
    warmup_steps=warmup_steps,
    lr_scheduler_type="linear",
    optim="adamw_torch",
    fp16=True,
    bf16=False,
    logging_steps=50,
    save_strategy="no",
    load_best_model_at_end=False,
    report_to="none",
    seed=SEED,
    data_seed=SEED,
    dataloader_num_workers=0,
    eval_accumulation_steps=1,
)
# Kaggle images can carry different supported Transformers versions.
argument_parameters = inspect.signature(TrainingArguments.__init__).parameters
strategy_parameter = (
    "eval_strategy" if "eval_strategy" in argument_parameters else "evaluation_strategy"
)
argument_values[strategy_parameter] = "no"
training_args = TrainingArguments(**argument_values)
trainer_values = dict(
    model=model,
    args=training_args,
    train_dataset=ds_tr,
    data_collator=collator,
)
trainer_parameters = inspect.signature(Trainer.__init__).parameters
if "processing_class" in trainer_parameters:
    trainer_values["processing_class"] = tokenizer
elif "tokenizer" in trainer_parameters:
    trainer_values["tokenizer"] = tokenizer
trainer = Trainer(**trainer_values)
started = time.monotonic()
train_result = trainer.train()
training_seconds = time.monotonic() - started
print("Training minutes:", round(training_seconds / 60, 1))
print("Completed refit epochs:", trainer.state.epoch)
print("No held-out score was computed for this full-data refit.")

## 7. Export final model and honest provenance
The final model is trained on all labels. The run configuration records data hashes, settings and the earlier v2 development score in a separately named provenance field. It does not report training performance as validation performance. Model weights remain in Kaggle output storage.


In [ ]:
require(trainer.state.epoch >= EPOCHS - 1e-6, "Training did not complete the selected epoch count.")
final_dir = WORK / "final_model"
trainer.save_model(str(final_dir))
tokenizer.save_pretrained(final_dir)
trainer.state.save_to_json(str(WORK / "trainer_state.json"))
weight_files = list(final_dir.glob("*.safetensors")) + list(final_dir.glob("pytorch_model*.bin"))
require((final_dir / "config.json").is_file(), "Final model config was not exported.")
require(
    any(path.is_file() and path.stat().st_size > 0 for path in weight_files),
    "Final model weights were not exported.",
)
run_config = {
    "approved_backbone": "FacebookAI/xlm-roberta-base",
    "experiment": "v3: full 14000-row refit, 512 tokens, 3 epochs, effective batch 16",
    "initialization": "fresh approved base checkpoint; not resumed from v2",
    "attached_model_path": str(MODEL),
    "seed": SEED,
    "labels": LABELS,
    "max_length": MAX_LENGTH,
    "epochs": EPOCHS,
    "completed_epochs": trainer.state.epoch,
    "learning_rate": LEARNING_RATE,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "gradient_accumulation": GRADIENT_ACCUMULATION,
    "weight_decay": 0.01,
    "warmup_steps": warmup_steps,
    "split_method": split_method,
    "fit_rows": len(tr),
    "validation_rows": 0,
    "refit_validation_macro_f1": None,
    "checkpoint_policy": "final epoch export only; no intermediate checkpoint selection",
    "training_seconds": training_seconds,
    "development_configuration_evidence": {
        "experiment": "v2",
        "fit_rows": 11200,
        "validation_rows": 2800,
        "validation_macro_f1": 0.9425011698287723,
        "note": "This score belongs to the earlier development model, not this full-data refit.",
    },
    "data_sha256": {
        name: hashlib.sha256((DATA / name).read_bytes()).hexdigest()
        for name in ["train.csv", "test.csv", "sample_submission.csv"]
    },
    "versions": {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "datasets": datasets.__version__,
    },
    "token_length_diagnostics": truncation_diagnostics,
}
(WORK / "run_config.json").write_text(json.dumps(run_config, indent=2), encoding="utf-8")
print("Saved final refit model and provenance:", WORK)
print("Refit held-out macro-F1: unavailable; all 14,000 labelled examples were used for training.")

## 8. Predict test rows and create the checked submission
Predictions are joined by ID into the sample's exact row order. The notebook rereads the output and checks the schema, row count, ID coverage, and labels. It does not consume a Kaggle submission attempt automatically.

In [ ]:
test_output = trainer.predict(ds_te, metric_key_prefix="test")
test_logits = test_output.predictions
if isinstance(test_logits, tuple):
    test_logits = test_logits[0]
test_logits = np.asarray(test_logits)
require(test_logits.shape == (len(test), len(LABELS)), "Wrong test prediction shape.")
require(np.isfinite(test_logits).all(), "Non-finite test logits.")
test_prediction = test_logits.argmax(axis=-1)
predictions = test[["id"]].copy()
predictions["label"] = [I2L[int(index)] for index in test_prediction]
require(predictions["id"].is_unique, "Duplicate prediction IDs.")
submission = sample[["id"]].merge(
    predictions, on="id", how="left", validate="one_to_one", sort=False
)
require(submission["label"].notna().all(), "Missing prediction after joining IDs.")
require(submission["id"].tolist() == sample["id"].tolist(), "Sample ID order changed.")
require(set(submission["label"]) <= set(LABELS), "Prediction contains an invalid label.")
submission_path = Path("/kaggle/working/submission.csv")
submission.to_csv(submission_path, index=False)
written = pd.read_csv(submission_path, dtype=str, keep_default_na=False)
require(list(written.columns) == ["id", "label"], "Written CSV has incorrect headers.")
require(len(written) == 6000 and written["id"].is_unique, "Written CSV has incorrect rows or IDs.")
require(written["id"].tolist() == sample["id"].tolist(), "Written CSV has incorrect ID order.")
require(set(written["label"]) <= set(LABELS), "Written CSV has invalid labels.")
require(written.equals(submission), "Written CSV did not round-trip exactly.")

np.save(WORK / "test_logits.npy", test_logits)
test_rows = predictions.copy()
for index, label in enumerate(LABELS):
    test_rows[f"logit_{label}"] = test_logits[:, index]
test_rows.to_csv(WORK / "test_predictions.csv", index=False)
print("READY:", submission_path)
print("Rows:", len(written), "Columns:", list(written.columns))
print("Predicted class counts (not a quality score):")
print(written["label"].value_counts().reindex(LABELS, fill_value=0).to_string())
print(
    "Full-data refit: no held-out validation score. Compare the eventual leaderboard score separately."
)

## 9. Review before submitting
1. Confirm all cells succeeded and the final cell prints `READY`.
2. Preserve this version, its provenance, final remote model and checked output CSV.
3. Submit the checked `submission.csv` only after team review; this notebook does not submit automatically.
4. Record the accepted competition receipt and public score separately from the earlier development score. Hidden test performance cannot be inferred from training loss.
5. Follow the organisers' notebook/team naming and code-access requirements. This title and team name are `RMIT_SGS_HaiQuayXe_C1`.

This run is reproducible from the approved model and competition CSV inputs. There are no external models, external training data, label heuristics or test-label access.
